# Análisis — Turgalium

De `data/raw/` a tablas y grupos de ejecución (AGENTS.md: todo lo de `data/derived/` se regenera desde aquí).

- **Subparticiones de configuración**: fingerprint del descubrimiento (`discovery/fingerprint.py`) — descriptivas.
- **Grupos de ejecución**: prueba de equivalencia en modo usuario (`experiments/equivalencia.sh`, DEVIATIONS 2026-09-26) — operativos; definen los `nodelists` de calibración, fases A/B, anclas y ligero.
- **Criterio R1** (RULES.md): distinguible si la diferencia de medianas supera el 5% **y** los IQR no se solapan. Un candidato se fusiona con la referencia si no es distinguible en **ninguna** de las tres métricas (STREAM Triad, latencia 8 B, bw 4 MiB). Si es distinguible: grupo propio con ≥8 nodos; si no, exclusión de los jobs multinodo y lista de exclusión.

In [ ]:
import hashlib, importlib.util, json, math, os, re, statistics
from collections import defaultdict
from pathlib import Path

REPO = Path(os.environ.get('CC_REPO', Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd())).resolve()
RAW_EQ = REPO/'data/raw/equivalence/turgalium'
DERIVED = Path(os.environ.get('CC_EQ_OUT', REPO/'data/derived/equivalence/turgalium'))
DERIVED.mkdir(parents=True, exist_ok=True)

med = lambda xs: statistics.median(xs)
def iqr(xs):
    xs = sorted(xs); n = len(xs)
    return (xs[n//4], xs[(3*n)//4]) if n >= 4 else None

# Un cmdlog por job; ambos hosts identifican el par, pero solo rank 0 imprime.
# No se deduplican hashes: dos rondas pueden producir exactamente la misma salida.
triad, lat, bw = {}, {}, {}
medidas = []
for cmdlog in sorted(RAW_EQ.glob('*/commands.jsonl')):
    celdas = defaultdict(list)
    for line in cmdlog.read_text().splitlines():
        j = json.loads(line)
        name = Path(j['cmd'][0]).name
        metric = ('lat' if name == 'osu_latency' else 'bw' if name == 'osu_bw'
                  else 'triad' if name.startswith('stream') else None)
        if metric:
            celdas[metric].append(j)
    for metric, entries in celdas.items():
        nodes = sorted({j['host'] for j in entries})
        if len(nodes) != (1 if metric == 'triad' else 2):
            raise ValueError(f'{cmdlog}: hosts ambiguos para {metric}: {nodes}')
        vals = []
        for j in entries:
            out = cmdlog.parent/'outputs'/f"{j.get('out_hash')}.out"
            if not out.exists():
                print(f'SIN SALIDA: {cmdlog.parent.name} {metric} {j["host"]}')
                continue
            raw = out.read_bytes()
            if hashlib.sha256(raw).hexdigest() != j.get('out_hash'):
                raise ValueError(f'hash no coincide: {out}')
            if j['exit_code'] != 0:
                continue
            txt = raw.decode(errors='replace')
            pattern = (r'^Triad:\s+([\d.]+)' if metric == 'triad'
                       else rf'^{8 if metric == "lat" else 4194304}\s+([\d.]+)')
            vals.extend(float(m.group(1)) for m in re.finditer(pattern, txt, re.M))
        if any(not math.isfinite(v) or v <= 0 for v in vals):
            raise ValueError(f'{cmdlog}: valores inválidos de {metric}')
        efectivos = vals[1:]  # calentamiento de CADA job/celda, como antes (R1)
        if efectivos:
            target = {'triad': triad, 'lat': lat, 'bw': bw}[metric]
            target.setdefault('|'.join(nodes), []).extend(efectivos)
        medidas.append({'job': cmdlog.parent.name, 'metric': metric, 'nodes': nodes,
                        'validas': len(vals), 'efectivas': len(efectivos)})
print(f'STREAM: {len(triad)} nodos | pares lat: {len(lat)} | pares bw: {len(bw)}')
for m in medidas:
    print(f"  job {m['job']} {m['metric']} {','.join(m['nodes'])}: {m['validas']} válidas, {m['efectivas']} efectivas")


In [ ]:
# Subparticiones de configuración por partición (fingerprint) y grupos de referencia M
# Mismo algoritmo de fingerprints; sin depender de derivados ignorados ni escribirlos.
_spec = importlib.util.spec_from_file_location('fingerprint', REPO/'discovery/fingerprint.py')
_fp = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_fp)
sp = _fp.build(_fp.load_nodes(str(REPO/'data/raw/discovery/turgalium'), False))
config = {}
M_NODES = {}   # referencia = la subpartición de configuración mayoritaria (derivada, no hardcodeada)
for part in ('cpu112c', 'cpu36c', 'volta', 'h100'):
    part_nodes = set(sp['partitions'][part]['nodes'])
    config[part] = [s for s in sp['subpartitions'] if set(s['nodes']) & part_nodes]
    maj = max(config[part], key=lambda s: s['size'])
    if maj['size'] >= 2:
        M_NODES[part] = set(maj['nodes'])

# estado SMT real por nodo (threads_per_core del descubrimiento)
smt = {}
for f in (REPO/'data/raw/discovery/turgalium').glob('*.json'):
    d = json.loads(f.read_text())
    if d.get('role') == 'node':
        smt[d['node']] = d['attributes']['threads_per_core']['real'].get('value')
for part in config:
    print(part, len(config[part]), 'subparticiones de configuración:', [(s['id'], s['size']) for s in config[part]])

# Mismo grupo de emplazamiento que el criterio previo: prefijo observado.
def mismo_switch(pair):
    a, b = pair.split('|')
    return a[:3] == b[:3]

def faltantes(part, node):
    # En el caso degenerado GPU no hay jobs multinodo: OSU sigue omitido declarado.
    required = ('triad', 'lat', 'bw') if part in M_NODES else ('triad',)
    return [metric for metric in required
            if not any(node in p.split('|') and values
                       and (metric == 'triad' or mismo_switch(p))
                       for p, values in {'triad': triad, 'lat': lat, 'bw': bw}[metric].items())]

no_probados = {part: [{'node': node, 'missing': faltantes(part, node)}
                     for node in sp['partitions'][part]['nodes'] if faltantes(part, node)]
               for part in config}
probados = {part: set(sp['partitions'][part]['nodes']) - {e['node'] for e in no_probados[part]}
            for part in config}


In [ ]:
# Veredicto R1: cada subpartición candidata contra la mayoritaria de su partición
def r1(distM, distC):
    """True = distinguible (R1); False = no distinguible; None = sin datos."""
    if not distM or not distC:
        return None
    medM, medC = med(distM), med(distC)
    if abs(medC - medM) / medM <= 0.05:
        return False
    qM, qC = iqr(distM), iqr(distC)
    if qM is None or qC is None:
        return True          # n pequeño: criterio conservador (R1, se declara)
    return qC[1] < qM[0] or qM[1] < qC[0]

def diferencia(reference, candidate):
    return round(100 * (med(candidate) / med(reference) - 1), 6) if reference and candidate else None

verdict = {}
for part in ('cpu112c', 'cpu36c'):
    M = M_NODES[part] & probados[part]
    triadM = [med(triad[n]) for n in triad if n in M]
    latM = [med(v) for p, v in lat.items() if set(p.split('|')) <= M and mismo_switch(p)]
    bwM = [med(v) for p, v in bw.items() if set(p.split('|')) <= M and mismo_switch(p)]
    res = {}
    for g in config[part]:
        nodes = [n for n in g['nodes'] if n in probados[part]]
        if set(nodes) == M:
            continue
        if not nodes:
            continue
        nodes_m = [n for n in nodes if n in triad]
        tC = [med(triad[n]) for n in nodes_m]
        pairsC = [p for p in lat if set(p.split('|')) <= set(nodes)
                  and p.split('|')[0][:3] == p.split('|')[1][:3]]   # proxy mismo switch
        if pairsC:
            lC = [v for p in pairsC for v in lat[p]]
            bC = [v for p in pairsC for v in bw.get(p, [])]
        else:  # 1 nodo: medido contra el nodo fijo de referencia
            lC = [x for p, v in lat.items() if set(p.split('|')) & set(nodes) and p.split('|')[0][:3] == p.split('|')[1][:3] for x in v]
            bC = [x for p, v in bw.items() if set(p.split('|')) & set(nodes) and p.split('|')[0][:3] == p.split('|')[1][:3] for x in v]
        res[g['id']] = {'nodes': nodes, 'size': len(nodes),
                        'difference_pct': {'triad': diferencia(triadM, tC),
                                           'lat': diferencia(latM, lC), 'bw': diferencia(bwM, bC)},
                        'r1': {'triad': r1(triadM, tC), 'lat': r1(latM, lC), 'bw': r1(bwM, bC)}}
    verdict[part] = res

for part in ('cpu112c', 'cpu36c'):
    M = M_NODES[part] & probados[part]
    print(f'== {part} ==')
    print('  dispersión interna M (Triad por nodo):',
          sorted(round(med(triad[n])) for n in triad if n in M))
    for gid, r in verdict[part].items():
        m = r['r1']
        etiqueta = 'fusiona' if (m['triad'] is False or m['lat'] is False or m['bw'] is False) and not any(m.values()) else None
        print(f"  subpart {gid} ({r['size']}n): triad={m['triad']} lat={m['lat']} bw={m['bw']}")

In [ ]:
# Grupos de ejecución, lista de exclusión y nodelists (salidas a data/derived/)
from datetime import datetime, timezone

metric_labels = {'triad': 'STREAM Triad', 'lat': 'lat (8 B)', 'bw': 'bw (4 MiB)'}

salida = {'cluster': 'turgalium',
          'generated_at': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
          'criterio': 'DEVIATIONS 2026-09-26 (grupos de ejecución); R1 sobre STREAM Triad, lat 8B, bw 4MiB en modo usuario',
          'partitions': {}}

for part in ('cpu112c', 'cpu36c', 'volta', 'h100'):
    if M_NODES.get(part) is None:   # caso degenerado (DEVIATIONS 2026-09-25): sin subpartición >=2 nodos
        deg = [sorted(set(s['nodes']) & probados[part]) for s in config[part]]
        deg = [g for g in deg if g]
        salida['partitions'][part] = {'execution_groups': deg, 'exclusion': [],
            'nota': 'sin mayoría: caracterización solo 1 nodo por subpartición; OSU pt2pt omitido y declarado'}
        if no_probados[part]:
            salida['partitions'][part]['no_probados'] = no_probados[part]
        print(f'== {part} == caso degenerado: {len(deg)} grupos de 1 nodo (OSU pt2pt omitido)')
        continue
    grupos, exclusion = [], []
    fusionados = sorted(M_NODES[part] & probados[part])             # la referencia siempre es grupo
    for gid, r in verdict[part].items():
        m = r['r1']
        medidos = [v for v in m.values() if v is not None]
        if len(medidos) != 3:
            raise ValueError(f'{part} subpartición {gid}: referencia insuficiente; no se puede decidir R1')
        if all(v is False for v in medidos):
            fusionados += r['nodes']               # se fusiona con la referencia
        elif len(r['nodes']) >= 8:
            grupos.append(sorted(r['nodes']))      # grupo propio multinodo
        else:
            exclusion.append({'nodes': r['nodes'], 'r1': m, 'difference_pct': r['difference_pct']})
    grupos.insert(0, sorted(set(fusionados)))

    # E-a (DEVIATIONS 2026-09-30): leave-one-out por nodo, exclusión directa.
    # Proxy de mismo grupo de switch = mismo prefijo de nodo, hasta que E-b a)
    # incorpore la topología real (declarada + inferida).
    excl_nodos = []
    for gi, g in enumerate(grupos):
        keep = []
        for n in g:
            tM = [med(triad[m]) for m in g if m in triad and m != n]
            tC = [med(triad[n])] if n in triad else []
            d_t = r1(tM, tC) if tC and tM else None
            mismos = [p for p in list(lat) + list(bw) if n in p.split('|') and p.split('|')[0][:3] == p.split('|')[1][:3]]
            lM = [x for p, v in lat.items() if set(p.split('|')) <= set(g) and n not in p.split('|') and p.split('|')[0][:3] == p.split('|')[1][:3] for x in v]
            lC = [x for p in mismos if p in lat for x in lat[p]]
            d_l = r1(lM, lC) if lM and lC else None
            bM = [x for p, v in bw.items() if set(p.split('|')) <= set(g) and n not in p.split('|') and p.split('|')[0][:3] == p.split('|')[1][:3] for x in v]
            bC = [x for p in mismos if p in bw for x in bw[p]]
            d_b = r1(bM, bC) if bM and bC else None
            if any(d is None for d in (d_t, d_l, d_b)):
                raise ValueError(f'{part} {n}: sin referencia leave-one-out; atribución no decidible')
            dist = [d_t, d_l, d_b]
            if dist and any(dist):
                excl_nodos.append({'node': n, 'r1': {'triad': d_t, 'lat': d_l, 'bw': d_b},
                                   'difference_pct': {'triad': diferencia(tM, tC),
                                                      'lat': diferencia(lM, lC), 'bw': diferencia(bM, bC)}})
            else:
                keep.append(n)
        if keep:
            grupos[gi] = keep
        else:
            grupos[gi] = []
    exclusion += excl_nodos

    # SMT (ajuste de la persona, DEVIATIONS 2026-09-30): particionado duro por
    # estado SMT — no se puede garantizar la homogeneidad entre nodos con y sin SMT.
    por_smt = {}
    for g in grupos:
        for n in g:
            por_smt.setdefault(bool(smt.get(n) == 2), []).append(n)
    grupos = [sorted(v) for k, v in sorted(por_smt.items()) if v]
    salida['partitions'][part] = {'execution_groups': grupos, 'exclusion': exclusion,
                                  'no_probados': no_probados[part]}
    print(f'== {part} ==')
    for i, g in enumerate(grupos):
        print(f'  grupo {i} ({len(g)} nodos): {",".join(g)}')
    for e in exclusion:
        nodos = e.get('nodes') or [e['node']]
        print(f'  EXCLUIDOS {",".join(nodos)}: r1={e["r1"]}')

    for e in no_probados[part]:
        print(f"  NO PROBADO {e['node']}: faltan {','.join(e['missing'])}")

(DERIVED/'grupos_ejecucion.json').write_text(json.dumps(salida, indent=2, ensure_ascii=False) + '\n')
for part in ('cpu112c', 'cpu36c', 'volta', 'h100'):
    for i, g in enumerate(salida['partitions'][part]['execution_groups']):
        (DERIVED/f'nodelist_{part}_{i}.txt').write_text(','.join(g) + '\n')
    # Una regeneración con menos grupos no deja nodelists antiguos utilizables.
    expected = {f'nodelist_{part}_{i}.txt' for i in range(len(salida['partitions'][part]['execution_groups']))}
    for stale in DERIVED.glob(f'nodelist_{part}_*.txt'):
        if stale.name not in expected:
            stale.unlink()
    with (DERIVED/f'exclusion_{part}.txt').open('w') as fh:
        for e in salida['partitions'][part]['exclusion']:
            nodos = e.get('nodes') or [e['node']]
            detalles = [f"{metric_labels[metric]} {e['difference_pct'][metric]:+.2f}%"
                        for metric, distinguished in e['r1'].items() if distinguished]
            fh.write(','.join(nodos) + ': ' + '; '.join(detalles) + '\n')
    (DERIVED/f'no_probados_{part}.txt').write_text(
        ''.join(f"{e['node']}: faltan {','.join(e['missing'])}\n"
                for e in salida['partitions'][part].get('no_probados', [])))
print('escrito en', DERIVED)

## Grupos con la prueba de equivalencia completa (T45)

Las celdas anteriores leen todos los cmdlogs de equivalencia, incluidos los pares
83703–83712 y 83713–83716. Los grupos actuales están en `grupos_ejecucion.json`.

- **cpu112c:** 13 nodos SMT-off y 2 SMT-on; b9p07 queda excluido por R1 en
  ancho de banda a 4 MiB (−39,29 % frente a su referencia leave-one-out).
  b9p01 no es distinguible pese a participar en ese par. Ningún nodo no probado.
- **cpu36c:** grupos de 20 y 17; siguen excluidos aac108 (lat/bw), aac111 y
  acc111 (Triad). Ningún nodo no probado.
- **volta/h100:** 8/2 grupos singleton; OSU pt2pt omitido declarado en el caso
  degenerado (DEVIATIONS 2026-09-25). No son grupos multinodo.

R1, referencia, leave-one-out y separación SMT conservados. Los nodos CPU sin
alguna medida válida intra-switch se listan aparte y no se admiten en grupos.
El estado anterior, que admitía b9p07 sin pares, queda como referencia histórica
en el primer commit de contenido de T45, reconstruido con el notebook antiguo.


In [ ]:
# Deduplicación del registro de la pasada de software (doble envío sw_*):
# clave (host, cmd, out_hash), se conserva la primera aparición.
raw_reg = REPO/'data/raw/discovery/turgalium/software/commands.jsonl'
seen, out_lines, dups = set(), [], 0
for ln in raw_reg.read_text().splitlines():
    try:
        e = json.loads(ln)
    except Exception:
        continue
    k = (e.get('host'), str(e.get('cmd')), str(e.get('out_hash')))
    if k in seen:
        dups += 1
        continue
    seen.add(k)
    out_lines.append(ln)
dst = REPO/'data/derived/discovery/turgalium'
dst.mkdir(parents=True, exist_ok=True)
(dst/'registro_software_dedup.jsonl').write_text('\n'.join(out_lines) + '\n')
print(f'registro software: {len(out_lines)} únicos ({dups} duplicados descartados de {len(out_lines)+dups})')

In [ ]:
# Inferencia de grupos de switch desde la matriz de latencias (E-b a):
# topología declarada NO disponible en turgalium (scontrol show topology vacío,
# verificado 2026-09-30); la real se infiere de las latencias entre pares:
# intra 0,92-1,04 µs frente a cruzados 1,36 µs (umbral 1,138 µs = media+sd intra x1,15).
pares = {p: med(v) for p, v in lat.items()}
intra = [v for p, v in pares.items() if p.split('|')[0][:3] == p.split('|')[1][:3]]
umbral_sw = (statistics.mean(intra) + statistics.stdev(intra)) * 1.15

# El corte observado coincide con el prefijo de nodo -> grupos de switch:
grupos_switch = {**{f'aac{i}': 'sw1' for i in range(101, 121)},
                 **{f'acc{i}': 'sw2' for i in range(101, 121)},
                 **{f'b9p{i:02d}': ('sw3' if i <= 16 else 'sin_clasificar') for i in range(1, 19)},  # b9p17/18 (h100): sin clasificar — partición distinta, pt2pt imposible entre particiones
                 **{f'aap{i:02d}': 'sin_clasificar' for i in range(1, 5)},   # sin medida pt2pt:
                 **{f'acp{i:02d}': 'sin_clasificar' for i in range(1, 5)}}     # grupo sin verificar
for tipo in (('aac', 'aac'), ('acc', 'acc'), ('aac', 'acc')):
    vals = [v for p, v in pares.items()
            if sorted(p.split('|'))[0][:3] == tipo[0] and sorted(p.split('|'))[1][:3] == tipo[1]]
    if vals:
        print(f'pares {tipo[0]}-{tipo[1]}: {len(vals)} pares, mediana {med(vals):.3f} µs')

out_sw = REPO/'data/derived/equivalence/turgalium/switch_groups.json'
out_sw.write_text(json.dumps({'umbral_us': round(umbral_sw, 3), 'switch_groups': grupos_switch,
                              'fuente': 'inferida de la matriz de latencias de la prueba de equivalencia; declarada no disponible'}, indent=2) + '\n')
print('switch_groups.json escrito')